# El toyset de MIL-CREDA — los dos términos de la fase 2 con sopesado por confianza y coeficientes fijos en 1, bajo el protocolo de dos fases de CREDA

Esto **entrena**. Corre MIL-CREDA —la atención β (Ecs. 16-17), el núcleo de bolsa, el
término global (Ecs. 32-37) y el término local con la correspondencia π (Ecs. 30-31)—
sobre las tres familias duras de `CREDA.toy_experiment.DURAS`, cortadas en bolsas por
`toy_bags`, y deja un artefacto en disco con la forma que `CREDA.toy_experiment.guardar`
le da al suyo.

**Para qué, en las palabras del operador:** mirar PRIMERO si MIL-CREDA funciona o no
bajo las condiciones que usa CREDA. Conectar este toyset con el banco de dígitos es
trabajo posterior y no está en esta corrida.

**Dos cosas separan esta corrida de la del banco de dígitos, y las dos son
declaraciones de este módulo, no detalles de implementación.**

- **La declaración de brazos es propia, y declara UNO.** `toy_run.TOY_ARMS` tiene un
  solo brazo: el adaptado. El piso —el brazo que no declaraba término de adaptación
  ninguno— lo **retiró el operador**, así que no hay escalera, no hay escalón, no hay
  piso y no hay margen contra un piso. No se lee `config.ARMS`: ésa es la declaración
  del banco de **dígitos**, de otro experimento, está fijada por un test que existe
  para atrapar la próxima reversión silenciosa de esa decisión. Y el protocolo tampoco
  es el de ellos: ese banco entrena **una** fase por `harness.run_one` y lee la
  **última** época, con el coeficiente saliendo de una rampa hacia un techo buscado;
  acá son **dos** fases, se lee el **mejor** paso, y el coeficiente es una constante.

  **Lo que cuesta un solo brazo, dicho acá y no al final.** Lo que esta corrida
  informa es un **acierto pelado**: no hay ningún número en este camino que diga
  contra qué leerlo. «Qué compran los dos términos de la fase 2» es una diferencia, y
  una diferencia necesita dos brazos. En particular el riesgo que `MIL-CREDA/AGREED.md`
  declara —que sin la transmisión fuente→blanco el brazo completo queda *por debajo
  del piso*— **ya no se puede medir acá**, y las dos lecturas que lo computaban (el
  escalón y el margen contra el piso) se fueron con el piso. Nada se dejó «de
  referencia»: una lectura retirada que sigue computando es un número que alguien
  cita. La celda de banderas imprime la declaración entera, con el motivo del brazo al
  lado y con la que NO se lee al lado de ésa.

  Las tres banderas de mecanismo del brazo adaptado —`adaptation`, `weighting`,
  `local`— quedaron **iguales** a las del brazo `G` de dígitos, y la celda de banderas
  lo **informa** en vez de esconderlo. Una versión anterior de esta sección sostenía lo
  contrario —que ningún brazo de dígitos tenía esta forma— y lo daba como la razón de
  la separación. Esa frase ya es falsa; lo que sostiene la separación es el pin y el
  protocolo.
- **El sopesado por confianza está ENCENDIDO, y lo que está apagado son los dos
  coeficientes de la Ec. (39).** El operador pidió la fase 2 «sin sopesado los dos
  loss», y una etapa anterior leyó *sopesado* como el sopesado por **confianza**
  (`w_j^t`, Ecs. 24/26/37) y declaró este brazo con `weighting: False`. Se refería a
  los **coeficientes de balance** de la Ec. (39): sin `λ_glob` y sin `λ_loc`. Así que
  la confianza vuelve a estar prendida —corren la `D` de la Ec. (26), los `ω_c` de la
  Ec. (37) y la agregación propia de la Ec. (38), que nunca estuvo detrás de esa
  bandera— y **no hay nada de la confianza apagado en ningún lado de este camino**.
- **Los dos coeficientes valen exactamente 1, fijos, sin rampa.** La Ec. (39) es
  `L = L_src + λ_glob·L_glob + λ_loc·L_loc`, y «sin los lambdas» significa ambos en 1.
  `harness.ramp` **no se llama** en este camino: un 1 fijo no es una rampa evaluada en
  su techo —la rampa arranca en 0.0 en el primer paso y se acerca al techo sin
  alcanzarlo—, así que pasarle `ceiling=1.0` habría dejado un cronograma que nadie
  declaró. Y por eso **cuatro de las ocho franjas declaradas salieron de la búsqueda**:
  `ceiling` ES `λ_glob`, `ceilingLocal` ES `λ_loc`, y los dos `rampDelta` son cómo
  crecía cada uno. Pasar cualquiera de esas cuatro llaves a un diccionario de
  hiperparámetros se **rechaza por nombre**, porque un `barrido.json` viejo las trae y
  quien las manda cree que decide algo.
- **Lo que cuesta encender el sopesado, medido por la corrida misma.** Los `ω_c` de la
  Ec. (37) son **sub-convexos** —suman a lo sumo uno— y acá las confianzas son bajas,
  así que el término global sale **dividido**. Con un techo buscado eso se absorbería;
  con el coeficiente fijo en 1 no, y la parte global que se lee en las tablas lleva ese
  factor adentro. `toy_run.weighting_cost` lo mide una vez por celda y cada fila lleva
  `weightingCostRatio`, `weightingCostOmegaSum` y `weightingCostGlobalUnweighted`.

  El factor **no** está escrito como número en ninguna parte de esta prosa, y eso es una
  corrección y no una omisión: una etapa anterior midió un rango y esa lectura **no se
  reproduce** acá, porque se tomó sobre un brazo ENTRENADO con `weighting: False` —otro
  θ_t, otras confianzas—. Un número escrito acá es una afirmación que nadie vuelve a
  medir; los dos rangos salen de `toy_run.WEIGHTING_COST["priorMeasurement"]`, que la
  celda de banderas imprime rotulado como anterior, y la lectura de ESTA corrida sale
  de las tablas, medida celda por celda.

El aparato es el de CREDA y está en `MIL_CREDA_Benchmark.toy_run`:

1. **Dos fases.** La fase 1 ajusta el encoder de la fuente, los parámetros de
   atención de la Ec. (15) y el cabezal sobre el término supervisado SOLO (Ec. 21),
   para temprano con el acierto de VALID de la fuente, **restaura los pesos de su
   mejor época** y los **congela**. Después θ_t arranca bit a bit idéntico a θ_s —una
   sola copia— y la fase 2 deja que el término global y el local, **cada uno entrando
   en la Ec. (39) con coeficiente fijo 1**, muevan θ_t y nada más. No se evalúa
   ninguna rampa.
2. **Un criterio para la parada de la fase 2 y para su búsqueda: el vigía,
   MINIMIZADO.** El vigía es el cabezal compuesto con el encoder del BLANCO, leído
   sobre la validación de la FUENTE, contra las etiquetas de la FUENTE. Cero
   etiquetas del blanco. Su argmax cae trivialmente en el paso 0 —θ_t arranca siendo
   θ_s—, así que el extremo informativo es el **argmin**.
3. **La fase 2 restaura los pesos del mejor paso al salir.** La razón es la de CREDA,
   textual: «quedarse con los últimos y sólo reportar el paso sería el mismo defecto
   que la fase 1 ya tenía.»
4. **`config.SOURCE_TO_TARGET_SYNC_STEPS` está retirado en este camino.** Dos fases
   significa que la fuente se congela y nada se transmite después de la única copia.
   El riesgo está declarado en `MIL-CREDA/AGREED.md` y **no se discute acá**: sin la
   transmisión, el brazo completo quedó POR DEBAJO del piso en dígitos. Si se
   reproduce acá, eso es un resultado y este cuaderno lo informa.

**Este cuaderno corre a escala de ENSAYO.** `toy_run.FULL` declara la escala que un
veredicto requiere —la de CREDA, medida— y `toy_run.PILOT` es esa misma configuración
con los conteos de repetición y los largos bajados y **nada más** distinto. Cada
registro que se escribe lleva el sello de la escala con que se midió, así que un
número de ensayo no puede viajar como resultado. **Nada acá libera la corrida
completa:** sólo una autorización explícita lo hace, y una verificación limpia o un
ensayo verde no son esa autorización.

## Dónde está el repositorio

In [1]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


## Las banderas de esta corrida, y las dos escalas

Lo primero que se imprime es **bajo qué se está midiendo**: la escala, la revisión, el
dispositivo, las familias, las normalizaciones, los brazos de la escalera, los
globales que se doblan, las **cuatro** franjas que se buscan y las **cuatro** que se
retiraron con el cronograma, y lo que cuesta encender el sopesado por confianza. Una
tabla que no dice bajo qué se midió no se puede leer, y esta celda es donde eso queda
dicho antes de que exista cualquier número.

**Tres cosas que hay que mirar acá y no después.** La primera: `config.CLASSES` vale
10 y estas familias tienen 3 clases, así que ésa se **dobla**. `config.BAGS_PER_CLASS`
y `config.INSTANCES_PER_BAG` valen 12 y 30, y este toyset adoptó **esos mismos dos
números** —son la regla del banco de dígitos, no una cifra propia—, así que para ellos
el doblez es un no-op y el código no lo trata como un caso aparte: un binding que
saltara un global cuando coincide sería un segundo camino que nada ejercita.
`config.py` no se toca en este camino, así que `toy_run.material_bound` los dobla y
los restaura en un `finally`. Es un global doblado y no un parámetro pasado: funciona
hasta que dos corridas se superponen en un mismo proceso, y para que eso se note en
vez de medirse, la entrada **se niega** si encuentra alguno de ellos fuera de su valor
declarado.

La segunda: `FRAC_TRAIN`/`FRAC_VALID` de CREDA son fracciones de los **puntos**, y acá
se parten **bolsas**. Con el número de bolsas por clase que la escala declara, la
fracción pedida no cae en un entero de bolsas, así que lo que se usa es la partición
entera más cercana con cada rol cubriendo cada clase. La celda imprime las dos cosas
—`partición pedida / realizada` y `bolsas por clase y rol`— porque el pedido no se
puede leer como la medición; ningún número de esa frase está escrito acá. Lo que sí
hay que leer en esos números es **en qué dirección cayó el redondeo**: `train` y
`valid` redondean y `test` es el complemento, así que el rol sobre el que descansa el
veredicto es el que **recibe** el resto, no el que está protegido de él.

La tercera: **las instancias por bolsa están declaradas, no derivadas.** Antes salían
de `points_por_dominio // bolsas_totales` en el sello mientras el binding doblaba
`config.INSTANCES_PER_BAG` a la bolsa más chica que encontraba en el sorteo, y los dos
números no coincidían. Ahora la bolsa es un número del programa —igual en las dos
escalas— y la medición pasó a ser una **guarda**: si el sorteo no puede pagar la bolsa
declarada, se niega antes de correr en vez de correr algo más chico. La consecuencia
está en la celda: `points_per_domain` dejó de ser una reducción usable, porque sólo
puede bajar hasta donde siga financiando 36 bolsas de 30.

In [2]:
# El repositorio lo resolvió la celda de arriba --- la de la forja, que viaja
# byte por byte --- y ésta sólo lo usa.
import dataclasses
import json
import sys
import textwrap
import time
from pathlib import Path

REPOSITORY = ROOT
sys.path.insert(0, str(REPOSITORY / "src"))

from MIL_CREDA_Benchmark import config, harness, toy_run

ESCALA = toy_run.PILOT          # nada acá libera `toy_run.FULL`
DISPOSITIVO = harness.resolve_device()
FAMILIAS = tuple(nombre for nombre, _ in toy_run.DURAS)
DESTINO = toy_run.destination(ESCALA, REPOSITORY)

print("repositorio:   ", REPOSITORY)
print("revisión:      ", config.REVISION)
print("dispositivo:   ", DISPOSITIVO)
print("escala:        ", ESCALA.name, "(ensayo)" if ESCALA.stamp()["isPilot"] else "")
print("destino:       ", DESTINO.relative_to(REPOSITORY))
print()
print("--- la escala declarada (FULL) contra la que corre (ESCALA) ---")
for campo in sorted(toy_run.REPETITION_FIELDS):
    lleno, corto = getattr(toy_run.FULL, campo), getattr(ESCALA, campo)
    print(f"  {campo:20s} completa {str(lleno):28s} ensayo {corto}")
print("  --- y todo lo que NO cambia entre las dos ---")
for campo in sorted({f.name for f in dataclasses.fields(toy_run.Scale)}
                    - toy_run.REPETITION_FIELDS - {"name"}):
    print(f"  {campo:20s} {getattr(ESCALA, campo)}")
print()
print("instancias por bolsa (derivadas):", ESCALA.instances_per_bag)
print("partición pedida / realizada:    ",
      json.dumps(ESCALA.realized_split()["requested"]), "/",
      json.dumps(ESCALA.realized_split()["realized"]))
print("bolsas por clase y rol:          ",
      json.dumps(ESCALA.realized_split()["bagsPerClass"]))
print()
print("familias:                ", list(FAMILIAS))
print("códigos de dominio:      ", json.dumps(toy_run.FAMILY_CODES))
print("normalizaciones:         ", json.dumps(toy_run.NORMALIZATIONS))
print("normalización ausente:   ", list(toy_run.NORMALIZATION_ABSENT),
      "(CREDA dibuja tres filas; `mlp.MLP_BACKBONES` declara dos)")
print()
print("--- la declaración de brazos de ESTE toyset, no `config.ARMS` ---")
print("brazos declarados:       ", list(toy_run.TOY_ARM_IDS),
      f"({len(toy_run.TOY_ARM_IDS)})")
for _brazo in toy_run.TOY_ARMS:
    print(f"  {_brazo['id']:4s} {_brazo['name']}")
    print(f"       adaptación={_brazo['adaptation']}  "
          f"sopesado={_brazo['weighting']}  local={_brazo['local']}  "
          f"unidad={_brazo['unit']}")
    for _linea in textwrap.wrap(_brazo["why"], 86):
        print(f"       {_linea}")
# No hay escalera, ni escalón, ni piso. La nota que lo dice viaja EN las
# banderas --- no es una frase de este cuaderno --- así que se imprime de ahí.
print("escalera / escalón / piso:")
for _linea in textwrap.wrap(
        toy_run.flags(ESCALA, DISPOSITIVO, FAMILIAS, [], 0.0)["armsNote"], 86):
    print(f"  {_linea}")
print()
print("la declaración que NO se lee acá, y la razón medida:")
print("  config.ARMS =", [_b["id"] for _b in config.ARMS],
      "--- el banco de DÍGITOS, otro experimento")
# Las tres banderas de mecanismo del brazo adaptado quedaron iguales a las de
# `G`. Se informa, no se esconde: la separación la sostienen el pin del banco
# de dígitos y el protocolo, no la ausencia de esta forma allá.
_ADAPTADO = toy_run.TOY_ARMS_BY_ID[toy_run.TOY_ADAPTED]
_FORMA = lambda _b: (_b["adaptation"], _b["weighting"], _b["local"], _b["unit"])
print("  ¿algún brazo de dígitos tiene las MISMAS tres banderas que el brazo",
      "adaptado de acá?", [_b["id"] for _b in config.ARMS
                           if _FORMA(_b) == _FORMA(_ADAPTADO)] or "ninguno")
print("  lo que difiere en cambio es el CRONOGRAMA:",
      f"ese brazo toma su coeficiente de harness.ramp con un techo buscado; "
      f"acá los dos valen {toy_run.FIXED_COEFFICIENT} fijos y no se llama a "
      f"ninguna rampa")
print()
print("--- qué gobierna exactamente `weighting: True` (encendido acá) ---")
for _clave, _texto in toy_run.WEIGHTING_GATES.items():
    print(f"  {_clave}:")
    for _linea in textwrap.wrap(_texto, 86):
        print(f"       {_linea}")
print()
print("--- lo que cuesta encender el sopesado por confianza ---")
for _clave in ("what", "whyItMatters", "measuredBy"):
    print(f"  {_clave}:")
    for _linea in textwrap.wrap(toy_run.WEIGHTING_COST[_clave], 86):
        print(f"       {_linea}")
print("  medición ANTERIOR --- otra etapa, brazo ENTRENADO sin sopesado;",
      "NO se reproduce acá. Las razones de esta corrida salen en las tablas:")
print("       ", json.dumps(
    {_k: (list(_v) if isinstance(_v, tuple) else _v)
     for _k, _v in toy_run.WEIGHTING_COST["priorMeasurement"].items()
     if _k != "measuredOn"}))
print()
print("--- lo que este toyset SÍ comparte con el banco de dígitos ---")
for _que, _porque in toy_run.SHARED_WITH_DIGIT_BENCHMARK.items():
    print(f"  {_que}:")
    for _linea in textwrap.wrap(_porque, 86):
        print(f"       {_linea}")
print()
print("globales doblados por `material_bound` (valor declarado -> doblado):")
for nombre in toy_run.BENT_GLOBALS:
    print(f"  config.{nombre:20s} {toy_run._DECLARED_GLOBALS[nombre]}")
print()
# Se leen de `searched_ranges`, que es de donde sale `banderas.json` ->
# `searchSplit.ranges`, y NO de `toy_run.SEARCH_RANGES`. Esa constante tiene
# sólo las franjas `*_RANGE` que declara `config`, así que imprimirla acá
# mostraba cuatro de las nueve dimensiones que se buscan: `lr1`/`weightDecay1`
# ya faltaban antes, y `dim`/`lr2`/`weightDecay2` se sumaron después. El
# registro estaba completo y este documento no.
_buscadas = {1: toy_run.searched_ranges("phase1"),
             2: toy_run.searched_ranges("phase2")}
_cuantas = sum(len(_f) for _f in _buscadas.values())
print(f"las {_cuantas} dimensiones que SE BUSCAN, y en qué fase cae cada una:")
for _fase, _franjas in _buscadas.items():
    for nombre, franja in _franjas.items():
        print(f"  fase {_fase}  {nombre:22s} {franja}")
print(f"  (fase 1: {len(_buscadas[1])} · fase 2: {len(_buscadas[2])})")
print(f"  `toy_run.SEARCH_RANGES` lista {len(toy_run.SEARCH_RANGES)} "
      f"({', '.join(toy_run.SEARCH_RANGES)}): son las franjas `*_RANGE` de "
      f"`config` y nada más, así que no es la cuenta de lo que se busca")
print("las cuatro que se RETIRARON con el cronograma de la Ec. (39):")
for nombre, _entrada in toy_run.RETIRED_RANGES.items():
    print(f"  retirada  {nombre:22s} {_entrada['range']}")
    print(f"            era {_entrada['was']}")
    for _linea in textwrap.wrap(_entrada["why"], 74):
        print(f"            {_linea}")
print(f"  (las dos listas parten las ocho franjas declaradas de `config`: "
      f"{len(toy_run.SEARCH_RANGES)} + {len(toy_run.RETIRED_RANGES)} = "
      f"{len(toy_run.SEARCH_RANGES) + len(toy_run.RETIRED_RANGES)})")
print(f"los dos coeficientes de la Ec. (39): {toy_run.FIXED_COEFFICIENT} fijos, "
      f"sin rampa --- y pasar una llave retirada se RECHAZA por nombre")
print()
print("por qué el corte está donde está:")
print(" ", toy_run.PHASE_ONE_SPLIT_REASON)
print("lo que la fase 1 NO busca:", json.dumps(toy_run.PHASE_ONE_NOT_SEARCHED))
# El piso de repeticiones para dar veredicto se retiró con el escalón: un
# veredicto es entre dos brazos y acá hay uno. Lo que SÍ se informa es el
# riesgo que ya no se puede medir, leído de las banderas y no escrito acá.
print()
print("el riesgo declarado que este camino ya NO puede contestar:")
for _linea in textwrap.wrap(
        toy_run.flags(ESCALA, DISPOSITIVO, FAMILIAS, [],
                      0.0)["syncRetired"]["riskNotMeasurableHere"], 86):
    print(f"  {_linea}")
print()
# EL CRITERIO DE LA FASE 2, leído de `toy_run.PHASE_TWO_CRITERION` y no escrito
# acá: un número tipeado en un párrafo no se puede volver a medir, y la mitad de
# estos vienen derivados de constantes de los DOS paquetes.
_CRIT = toy_run.PHASE_TWO_CRITERION
print("--- el criterio de la fase 2: se lee una vez por PASADA, no por paso ---")
for _linea in textwrap.wrap(_CRIT["what"], 86):
    print(f"  {_linea}")
print(f"  pasos de una pasada (declarado) {_CRIT['stepsPerEpochDeclared']}"
      f"   <- {_CRIT['stepsPerEpochDeclaredDerivation']}")
print(f"  la señal: {_CRIT['signal']}, {_CRIT['direction'].upper()}")
for _linea in textwrap.wrap(_CRIT["signalIs"], 82):
    print(f"       {_linea}")
for _linea in textwrap.wrap(_CRIT["signalChange"], 82):
    print(f"       {_linea}")
print(f"  mínimo  {_CRIT['minEpochs2']} pasadas   "
      f"{_CRIT['minEpochs2Is'].upper()}  (en CREDA son "
      f"{_CRIT['minEpochs2InCreda']}, y los dos valores viajan sin igualarse)")
# POR QUE difieren, que es la unica parte de esto que sigue abierta: los dos
# salieron del mismo piso en pasos sobre pasadas de largo distinto, y no
# igualarlos es una decision sobre la guarda y no sobre la unidad. Se imprime
# porque una decision abierta que no se ve no se toma.
for _linea in textwrap.wrap(_CRIT["minEpochs2WhyTheyDiffer"], 82):
    print(f"    {_linea}")
for _linea in textwrap.wrap(_CRIT["minEpochs2Bites"], 82):
    print(f"       {_linea}")
print(f"  paciencia: {_CRIT['patience']}")
for _linea in textwrap.wrap(_CRIT["patienceWhy"], 82):
    print(f"       {_linea}")
for _linea in textwrap.wrap(_CRIT["phaseOneStillStops"], 82):
    print(f"       {_linea}")
print(f"  el techo pasó a EPOCAS el 2026-10-19: epochs_2")
print(f"  el techo se cuenta en {_CRIT['budgetUnit'].upper()}")
for _linea in textwrap.wrap(_CRIT["budgetWhy"], 82):
    print(f"       {_linea}")
print("  lo que forzó el cambio, medido en CREDA a 200 pasos:")
for _fam, _lectura in _CRIT["budgetChangeMeasured"]["inCredaAt200Steps"].items():
    print(f"       {_fam:10s} {_lectura}")
for _linea in textwrap.wrap(_CRIT["budgetChangeMeasured"]["reading"], 82):
    print(f"       {_linea}")
print(f"  a escala completa: {_CRIT['atTheFullScale']['epochs2']} épocas = "
      f"{_CRIT['atTheFullScale']['stepsItHolds']} pasos, "
      f"{_CRIT['atTheFullScale']['stepsWithoutARead']} pasos SIN lectura")
print(f"  a escala de ensayo: {ESCALA.epochs_2} epocas = "
      f"{ESCALA.epochs_2 * _CRIT['stepsPerEpochDeclared']} pasos, "
      f"0 pasos SIN lectura (cero por construccion)")
print("  la asimetría contra CREDA, que viene del MATERIAL y no de una elección:")
_ASIM = _CRIT["asymmetryAgainstCreda"]
for _linea in textwrap.wrap(_ASIM["whyTheyDiffer"], 82):
    print(f"       {_linea}")
print(f"       pasadas en el techo:   acá {_ASIM['epochsInTheBudgetHere']}  "
      f"CREDA {_ASIM['epochsInTheBudgetInCreda']}")
for _linea in textwrap.wrap(_ASIM["reading"], 82):
    print(f"       {_linea}")
for _linea in textwrap.wrap(_ASIM["reading"], 82):
    print(f"       {_linea}")
print()
# EL ORÁCULO. Es la única lectura de este camino que toca `y_t`, así que su
# declaración se imprime entera en vez de resumirse.
print("--- el oráculo: la serie del blanco por paso, y qué NO hace ---")
_SER = toy_run.TARGET_SERIES
print(f"  campo {_SER['field']}   rol {_SER['role']}   cadencia {_SER['cadence']}"
      f"   unidad {_SER['unit']}")
print(f"  lee etiquetas del blanco: {_SER['readsTargetLabels']}   "
      f"entra en algún camino de selección: {_SER['entersNoSelectionPath'] is False}")
for _clave in ("why", "whyTheTestRole", "comparedWithCreda"):
    print(f"  {_clave}:")
    for _linea in textwrap.wrap(_SER[_clave], 82):
        print(f"       {_linea}")

/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repositorio:    /Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation
revisión:       research-concept-r34.md
dispositivo:    mps
escala:         pilot (ensayo)
destino:        MIL-CREDA/Results/Toyset/pilot

--- la escala declarada (FULL) contra la que corre (ESCALA) ---
  epochs_1             completa 100                          ensayo 6
  epochs_2             completa 100                          ensayo 4
  patience_1           completa 30                           ensayo 3
  points_per_domain    completa 4050                         ensayo 2835
  report_seeds         completa (0, 1, 2, 3, 4, 5, 6, 7, 8, 9) ensayo (0,)
  search_seeds         completa (100, 101, 102)              ensayo (100,)
  trials_1             completa 25                           ensayo 2
  trials_2             completa 25                           ensayo 2
  --- y todo lo que NO cambia entre las dos ---
  bags_per_class       27
  classes              3
  frac_train           0.6
  frac_vali

## El costo de una celda, antes de comprometer la grilla

Una estimación del costo es más barata que el costo, y va acá —primer paso— y no como
una frase en una conversación. Se cronometra **una celda**: una familia, una
normalización, una semilla, el único brazo declarado sobre la fase 1 que la celda
entrena una sola vez. De ahí salen dos **tasas medidas** —lo que cuesta una época de fase 1 y lo que
cuesta un paso de fase 2— y el pronóstico se arma contando el trabajo que cada bucle
hace, no multiplicando las razones de repetición entre sí.

**Ese producto fue un defecto y queda anotado**, porque parecía un pronóstico: los
conteos de ensayos multiplican las **búsquedas** y las semillas de reporte multiplican
las **celdas**, así que multiplicarlos todos juntos sobreestima por órdenes de
magnitud. La versión anterior de esta celda pronosticó miles de horas a partir de una
celda de dos segundos. El número que se imprime ahora es una **cota superior** y lo
dice: las dos paradas tempranas cortan antes del largo declarado, así que los conteos
de épocas y de pasos son topes.

**Acá también se mide la escala del núcleo**, y es lo que corrigió este módulo. La
constante `config.KERNEL_SIGMA` se midió en dígitos bajo resnet18 y no significa nada
acá: `KERNEL_SIGMA_RANGE` es esa constante por una década para cada lado, así que
arrastra la escala equivocada. La celda imprime qué lee esa constante sobre estas
familias. La grilla se centra en una **distancia medida**, sobre los multiplicadores
que `toy_run.SIGMA_MULTIPLIERS` declara, y el valor del núcleo en el mediano depende
**sólo** del multiplicador —`exp(-1/(2m²))`—, así que los cuatro puntos caen en banda
por construcción, cualquiera sea la distancia que se mida.

Lo que hay que mirar es que la distancia medida es la **de adentro de una bolsa** y no
la del montón. Una bolsa es una banda de UNA clase, así que su dispersión es mucho
menor que la del conjunto; medido acá, difieren por un factor que la celda imprime. Una
sigma centrada en el mediano del montón deja el núcleo de adentro de la bolsa casi
saturado y los pesos de la Ec. (16) a una fracción ínfima del uniforme —la celda
imprime cuál—: el mecanismo queda numéricamente apagado mientras todas las banderas
dicen que está prendido.

Y hay una consecuencia de la Decisión 1 que se informa en vez de arreglarse: **una
sola sigma tiene dos trabajos**. El núcleo de instancia lee distancias de adentro de
una bolsa; el Gram de bolsas de las Ecs. (32-37) lee distancias entre bolsas, y las dos
escalas difieren. Ningún valor único está en banda para las dos, así que la franja que
se busca abarca las dos y es la búsqueda la que elige el canje.

In [3]:
# Una celda, cronometrada, con los hiperparámetros DECLARADOS --- antes de que
# ninguna búsqueda haya elegido nada. Lo que se mide acá es el costo y la escala
# del núcleo, no un resultado.
_familia = toy_run.build_family(FAMILIAS[0], int(ESCALA.report_seeds[0]), ESCALA)
_t = time.perf_counter()
_celda = toy_run.run_cell(_familia, "batchnorm", int(ESCALA.report_seeds[0]),
                          ESCALA, toy_run.declared_hyper(ESCALA), DISPOSITIVO)
print(f"una celda ({_familia.nombre}/batchnorm/semilla "
      f"{ESCALA.report_seeds[0]}, {len(_celda['rows'])} brazos): "
      f"{_celda['elapsedSeconds']:.2f} s")

_costo = toy_run.cost_report([_celda["rates"]], ESCALA,
                             _celda["elapsedSeconds"], families=len(FAMILIAS))
print(f"  una época de fase 1: {_costo['secondsPerPhase1Epoch']:.3f} s")
print(f"  un paso de fase 2:   {_costo['secondsPerPhase2Step']:.3f} s")
print()
print("el trabajo que hace un barrido, contado sobre los bucles que corren")
print("  (los factores NO se multiplican entre sí: los ensayos multiplican las")
print("   búsquedas y las semillas de reporte multiplican las celdas)")
for _e, _c in (("a esta escala", _costo["workHere"]),
               ("a escala completa", _costo["workAtFullScale"])):
    print(f"  {_e:18s} fase 1: {_c['phase1Trainings']:>5} entrenamientos "
          f"({_c['phase1EpochCeiling']:>7} épocas tope)   "
          f"fase 2: {_c['phase2Runs']:>5} corridas "
          f"({_c['phase2StepCeiling']:>7} pasos tope)")
print(f"  escala de instancias por bolsa: x{_costo['instanceScale']:.1f}")
print(f"pronóstico de la escala completa: "
      f"{_costo['forecastFullHours']:.1f} h  (cota superior)")
print("  advertencia:", _costo["caveat"])
print()

_e = _celda["sigma"]["scales"]
print("la escala del núcleo, medida sobre el encoder tal como se construye:")
print(f"  mediano de adentro de una bolsa (Ec. 14 / consenso de la Ec. 15): "
      f"{_e['withinBag']:.4f}")
print(f"  mediano entre bolsas (Gram de las Ecs. 32-37):                   "
      f"{_e['betweenBags']:.4f}")
print(f"  mediano del montón (lo que una calibración ingenua habría usado): "
      f"{_e['pooled']:.4f}")
print(f"  razón entre los dos trabajos de UNA sigma: "
      f"{_celda['sigma']['scaleRatio']:.2f}x")
print(f"  dispersión de los pesos de la Ec. (16) contra el uniforme "
      f"{_e['uniformWeight']:.4f}: {_e['weightSpread']:.2e} "
      f"({_e['weightSpreadRelativeToUniform'] * 100:.2f}% del uniforme)")
print()
print(f"  la sigma declarada ({_celda['sigma']['declaredConstant']:.3f}) lee un "
      f"núcleo de {_celda['sigma']['declaredKernelWithinBag']:.6f} adentro de una "
      f"bolsa --- banda {_celda['sigma']['band']}")
print("  la grilla centrada en la distancia medida:")
for _p in _celda["sigma"]["points"]:
    print(f"    x{_p['multiplier']:<5} sigma {_p['sigma']:.4f}  "
          f"núcleo adentro {_p['kernelWithinBag']:.3f}  "
          f"entre bolsas {_p['kernelBetweenBags']:.3f}  "
          f"{'en banda' if _p['inBand'] else 'FUERA de banda'}")
print(f"  franja buscada (abarca las dos escalas): "
      f"({_celda['sigma']['range'][0]:.4f}, {_celda['sigma']['range'][1]:.4f})")
print()
print("  ", _celda["sigma"]["oneBandwidthTwoJobs"])

una celda (lunas/batchnorm/semilla 0, 1 brazos): 8.02 s
  una época de fase 1: 0.123 s
  un paso de fase 2:   0.086 s

el trabajo que hace un barrido, contado sobre los bucles que corren
  (los factores NO se multiplican entre sí: los ensayos multiplican las
   búsquedas y las semillas de reporte multiplican las celdas)
  a esta escala      fase 1:    36 entrenamientos (    216 épocas tope)   fase 2:    27 corridas (   1728 pasos tope)
  a escala completa  fase 1:   792 entrenamientos (  79200 épocas tope)   fase 2:   765 corridas (1224000 pasos tope)
  escala de instancias por bolsa: x1.0
pronóstico de la escala completa: 32.0 h  (cota superior)
  advertencia: an upper bound, not a measurement: both early stops end a phase before its declared length, so the epoch and step counts are ceilings, and the per-epoch cost is scaled linearly in instances per bag, which is an assumption about the encoder and not a measurement of it

la escala del núcleo, medida sobre el encoder tal como se con

## Las dos búsquedas, y después el único brazo

Las dos búsquedas son las de CREDA en forma: `TRIALS_1` ensayos para la fase 1 —una
configuración por familia, **compartida** por todas las normalizaciones que la
celda de banderas imprime, porque hiperparámetros por fila confundirían el eje de
la normalización con los hiperparámetros— y `TRIALS_2` para la fase 2, con la fase 1 congelada y entrenada una
sola vez por semilla de búsqueda. Las semillas de búsqueda son **disjuntas** de las que
reportan.

**La fase 2 busca UNA dimensión y antes buscaba cinco.** `ceiling`, `ceilingLocal`,
`rampDelta` y `rampDeltaLocal` eran cuatro de esas cinco y las cuatro son el
cronograma de la Ec. (39), retiradas con él. Queda `tauLocal`, que llega a la
correspondencia de las Ecs. (30)-(31) por la temperatura de `total_correspondence` y a
nada más. `TRIALS_2` no bajó, así que cada ensayo cae ahora sobre la única dimensión
que hace algo.

**Ninguna de las dos mira una etiqueta del blanco.** La de fase 1 maximiza el acierto
de VALID de la fuente y rompe empates con la pérdida de VALID; la de fase 2 minimiza el
**vigía**, que es el mismo número que usa la parada. Que la búsqueda y la parada
compartan criterio es el punto: una búsqueda que minimizara otra cosa estaría eligiendo
una configuración para una regla de parada que lee otro número. El acierto del blanco de
cada ensayo se **registra y no se lee** por el criterio, para que después se pueda
contestar cuál de las dos lecturas habría ordenado mejor —y eso no se puede si sólo se
guardó el ganador.

Después corre el **único brazo declarado** sobre las semillas de reporte. **No hay
escalera y no hay escalón**: el piso lo retiró el operador, y lo que queda es un
acierto con nada al lado contra qué leerlo. Esa ausencia es la decisión tomada, no una
pieza que falte, y este cuaderno la informa en vez de llenarla con una comparación
inventada.

La fase 1 se sigue entrenando **una sola vez** por celda y cargándose en el brazo que
la fase 2 mueve. Con un brazo eso no cambia ningún número —no hay con quién
compartirla— y se conserva porque es una propiedad del protocolo y no del número de
brazos: una versión que entrenara adentro del bucle volvería a ajustar la fase 1 por
brazo en el momento en que se declare un segundo, y nada lo diría.

**Qué se puede y qué no se puede leer de esta corrida.** Se puede leer si la fase 1
aprende la tarea en su propio dominio —el acierto de la FUENTE contra el azar, que con
tres clases es 1/3— y se puede leer cuánto manda cada término de la fase 2 en el
objetivo. No se puede leer qué compran esos términos, porque comprar es una diferencia.

In [4]:
# Las dos búsquedas y el barrido, a escala de ensayo. Esto es la corrida.
_t = time.perf_counter()
BARRIDO = toy_run.sweep(ESCALA, DISPOSITIVO, FAMILIAS)
print(f"\nbarrido completo: {time.perf_counter() - _t:.1f} s")
print(f"celdas: {len(BARRIDO['cells'])}  "
      f"filas: {len(BARRIDO['rows'])}  "
      f"brazos: {len(toy_run.TOY_ARM_IDS)} (sin escalones: no hay par que restar)")
print()
print("lo que cada búsqueda eligió, por familia:")
for _f in FAMILIAS:
    print(f"  {_f}")
    print(f"    fase 1: {json.dumps({k: (round(v, 5) if isinstance(v, float) else v) for k, v in BARRIDO['hp1'][_f].items()})}")
    print(f"      {BARRIDO['records1'][_f]['trials']} ensayos, "
          f"mejor acierto de VALID {BARRIDO['records1'][_f]['bestAccuracy']:.4f}, "
          f"{BARRIDO['records1'][_f]['tied']} empatados, "
          f"criterio: {BARRIDO['records1'][_f]['criterion']}")
    print(f"    fase 2: {json.dumps({k: round(v, 5) for k, v in BARRIDO['hp2'][_f].items()})}")
    print(f"      {BARRIDO['records2'][_f]['trials']} ensayos, "
          f"max(cef) {BARRIDO['records2'][_f]['bestCef']:.4f}, "
          f"criterio: {BARRIDO['records2'][_f]['criterion']}")
    # El vigía al lado y rotulado: desde el 2026-10-16 NO decide nada, y
    # publicar su valor bajo las palabras «min(vigía)» --- que es lo que esta
    # celda decía --- es un número con la etiqueta del criterio que lo
    # reemplazó.
    print(f"      el vigía del elegido: "
          f"{BARRIDO['records2'][_f]['bestWatchman']:.4f} "
          f"(registrado, NO leído por el criterio)")
    print(f"      acierto del blanco por ensayo (registrado, NO leído por el "
          f"criterio): "
          f"{[round(t['targetAccuracy'], 4) for t in BARRIDO['records2'][_f]['perTrial']]}")

  lunas


    lunas    phase 1: 2 trials, acc 0.8222, 1 within 0.01 -> the valid loss breaks the tie (0.0219)


    lunas    phase 2: 2 trials, max(cef) 2.8512


      lunas/ninguna/seed 0: 5.1s


      lunas/batchnorm/seed 0: 5.4s


...

    rollo    phase 1: 2 trials, acc 1.0000, 2 within 0.01 -> the valid loss breaks the tie (0.0128)


    rollo    phase 2: 2 trials, max(cef) 2.4458


      rollo/ninguna/seed 0: 6.1s


      rollo/batchnorm/seed 0: 5.9s


      rollo/layernorm/seed 0: 5.3s

barrido completo: 134.5 s
celdas: 9  filas: 9  brazos: 1 (sin escalones: no hay par que restar)

lo que cada búsqueda eligió, por familia:
  lunas
    fase 1: {"lr1": 0.00443, "weightDecay1": 0.00038, "attentionGamma": 0.60276, "attentionTemperature": 1.22961, "dim": 64, "sigmaMultiplier": 0.5, "kernelSigma": 0.1761}
      2 ensayos, mejor acierto de VALID 0.8222, 1 empatados, criterio: max(sourceValidAccuracy), ties by validLoss
    fase 2: {"tauLocal": 1.22961, "lr2": 0.00019, "weightDecay2": 0.00017}
      2 ensayos, max(cef) 2.8512, criterio: max(cef)
      el vigía del elegido: 0.7778 (registrado, NO leído por el criterio)
      acierto del blanco por ensayo (registrado, NO leído por el criterio): [0.537, 0.7593]
  toroide
    fase 1: {"lr1": 0.00443, "weightDecay1": 0.00038, "attentionGamma": 0.60276, "attentionTemperature": 1.22961, "dim": 64, "sigmaMultiplier": 0.5, "kernelSigma": 0.41555}
      2 ensayos, mejor acierto de VALID 1.0000, 2 emp

## Las filas, el acierto de la fuente primero, y lo que no hay contra qué leer

**La pregunta por la que esta corrida existe se contesta en la columna del acierto de
la FUENTE, y va primero.** Con tres clases, el azar es 1/3. Si la fase 1 no supera el
azar *en su propio dominio*, ningún número del blanco significa nada: no hay una
representación que transferir. A cuatro bolsas por clase la fase 1 entrenaba sobre **6
bolsas** y el acierto de la fuente salió 1/3 o 0 en las doce celdas medidas; a doce
bolsas por clase entrena sobre **21**. Lo que la tabla de abajo contesta es si eso
alcanzó.

Una fila por brazo —hay uno—, con la dispersión al lado de la media. **No hay
veredicto y no hay escalón**: comparar es restar dos brazos, el piso lo retiró el
operador, y esta corrida informa un acierto pelado con nada al lado contra qué leerlo.
No se puso otra comparación en ese lugar: una lectura inventada para tapar el hueco
contesta una pregunta que nadie hizo.

Lo que **sí** se lee al lado del acierto son las dos partes del objetivo —cuánto manda
el término global y cuánto el local— porque «el término no hizo nada» y «el término no
tenía peso» imprimen el mismo cero si no se separan.

**Y antes de leer esas dos partes, el factor que lleva adentro la parte global.** Los
`ω_c` de la Ec. (37) son sub-convexos y las confianzas de estas familias son bajas, así
que encender el sopesado por confianza **divide** el término global —la columna `razón`
de abajo es `global(con sopesado) / global(sin sopesado)` medida en cada celda por
`toy_run.weighting_cost`—. Con un `λ_glob` buscado eso se absorbería; con el
coeficiente fijo en 1 no se absorbe, así que una parte global chica frente a la local es
en parte ese factor y no sólo la matemática.

**La resolución de una celda, que es lo que doce bolsas por clase compró.** Con la
partición 7/2/3 el rol de evaluación tiene **3 bolsas por clase, 9 en total**, así que
el acierto de una celda toma uno de **10** valores y la diferencia más fina medible es
de 11 puntos porcentuales. A cuatro bolsas por clase la partición era 2/1/1: **una**
bolsa por clase, 3 en total, cuatro valores posibles y 33 puntos porcentuales de
resolución. Eso no cambia con la escala —`points_per_domain` mueve cuántas instancias
tiene una bolsa, no cuántas bolsas hay—; **lo que la escala completa agrega son
semillas y nada más**. Las normalizaciones son las mismas en las dos escalas:
`toy_run.NORMALIZATIONS` no es un campo de repetición, y cuántas hay lo imprime
la celda de banderas de arriba en vez de decirlo esta frase.

**Y las normalizaciones ahora son las tres de CREDA.** La tercera es
`layernorm`, y está por una razón que no es simetría con la figura de al lado:
es el **control** que separa «normalizar ayuda» de «reestimar ayuda». BatchNorm
hace las dos cosas a la vez —normaliza y guarda corridas que el dominio nuevo
puede reestimar, que es lo que AdaBN es—; LayerNorm normaliza y **no guarda
nada**, así que AdaBN no tiene escalón ahí y la celda lo dice en vez de poner un
número. Sin esa fila, una fila de BatchNorm que lee por encima de la fila sin
normalización no se puede atribuir a ninguna de las dos cosas.

In [5]:
def _pct(x):
    return f"{x * 100:6.2f}%"


# El acierto de la FUENTE primero y en su propia tabla: si la fase 1 no supera
# el azar en su propio dominio, ningún número del blanco significa nada. El azar
# se calcula de las clases que la escala declara, no escrito acá.
_AZAR = 1.0 / ESCALA.classes
print(f"PRIMERO: ¿la fase 1 supera el azar EN SU PROPIO DOMINIO? "
      f"azar = 1/{ESCALA.classes} = {_pct(_AZAR)}")
print(f"  (fase 1 entrena sobre "
      f"{ESCALA.realized_split()['bagsPerClass']['train'] * ESCALA.classes} "
      f"bolsas de entrenamiento; el acierto se lee sobre "
      f"{ESCALA.realized_split()['bagsPerClass']['test'] * ESCALA.classes} "
      f"bolsas de evaluación de la fuente)")
print()
print(f"  {'familia':9s} {'norm':10s} {'brazo':6s} {'acierto fuente':>16s} "
      f"{'¿supera el azar?':>18s}")
_SUPERAN = 0
_CELDAS = 0
for _f in FAMILIAS:
    for _n in toy_run.NORMALIZATIONS:
        for _a in toy_run.TOY_ARM_IDS:
            _s = BARRIDO["rows"][toy_run.key(_f, _n, _a)]["summary"]
            _m = _s["sourceAccuracy"]["mean"]
            _CELDAS += 1
            _SUPERAN += int(_m > _AZAR)
            print(f"  {_f:9s} {_n:10s} {_a:6s} "
                  f"{_pct(_m)} ± {_s['sourceAccuracy']['stdev']:.3f} "
                  f"{('sí' if _m > _AZAR else 'NO'):>18s}")
print()
print(f"{_SUPERAN} de {_CELDAS} celdas superan el azar en la fuente.")
if not _SUPERAN:
    print("  Ninguna. Eso es un resultado sobre el protocolo a esta escala, no")
    print("  una falla de este cuaderno: sin una representación que la fuente")
    print("  aprenda, no hay nada que transferir y ningún número del blanco")
    print("  significa nada.")
print()
print(f"el acierto del BLANCO al lado, sobre las "
      f"{len(ESCALA.report_seeds)} semillas de reporte --- y nada contra qué")
print("leerlo: el piso se retiró, así que no hay escalón ni margen")
print()
for _f in FAMILIAS:
    for _n in toy_run.NORMALIZATIONS:
        print(f"{_f} / {_n}")
        print(f"  {'brazo':6s} {'acierto blanco':>16s} {'acierto fuente':>16s} "
              f"{'parte adapt.':>14s} {'cef mejor':>10s} {'época mejor':>12s} "
              f"{'vigía':>8s}")
        for _a in toy_run.TOY_ARM_IDS:
            _fila = BARRIDO["rows"][toy_run.key(_f, _n, _a)]
            _r = _fila["runs"][0]
            _s = _fila["summary"]
            print(f"  {_a:6s} "
                  f"{_pct(_s['targetAccuracy']['mean'])} ± "
                  f"{_s['targetAccuracy']['stdev']:.3f} "
                  f"{_pct(_s['sourceAccuracy']['mean'])} ± "
                  f"{_s['sourceAccuracy']['stdev']:.3f} "
                  f"{_s['adaptationShare']['mean']:14.4f} "
                  f"{_r['cefBest']:10.4f} "
                  f"{_r['phase2BestEpoch']:>12d} "
                  f"{str(round(_r['watchmanBest'], 4))[:8]:>8s}")
        # Las dos partes del objetivo, aparte: «el término no hizo nada» y «el
        # término no tenía peso» son el mismo número si se suman. Y al lado, el
        # factor que la parte global lleva adentro --- medido en esta celda, no
        # citado: con el coeficiente fijo en 1 no hay un λ_glob que lo absorba.
        print(f"  {'brazo':6s} {'parte global':>14s} {'parte local':>14s} "
              f"{'razón sopes.':>14s} {'Σω (Ec. 37)':>14s}")
        for _a in toy_run.TOY_ARM_IDS:
            _r = BARRIDO["rows"][toy_run.key(_f, _n, _a)]["runs"][0]
            print(f"  {_a:6s} {_r['contributionGlobal']:14.6f} "
                  f"{_r['contributionLocal']:14.6f} "
                  f"{_r['weightingCostRatio']:14.4f} "
                  f"{_r['weightingCostOmegaSum']:14.4f}")
        print()

print("resolución de una celda: el rol de evaluación tiene "
      f"{ESCALA.realized_split()['bagsPerClass']['test'] * ESCALA.classes} bolsas, "
      f"así que el acierto sólo puede tomar "
      f"{ESCALA.realized_split()['bagsPerClass']['test'] * ESCALA.classes + 1} "
      "valores distintos.")
print("bolsas por clase y rol:", json.dumps(ESCALA.realized_split()["bagsPerClass"]),
      "--- partición pedida",
      json.dumps(ESCALA.realized_split()["requested"]))
print("instancias por bolsa (DECLARADAS):", ESCALA.instances_per_bag)
print()
print("lo que esta corrida NO puede contestar, leído de las banderas:")
for _linea in textwrap.wrap(BARRIDO["flags"]["armsNote"], 86):
    print(f"  {_linea}")
for _linea in textwrap.wrap(
        BARRIDO["flags"]["syncRetired"]["riskNotMeasurableHere"], 86):
    print(f"  {_linea}")


PRIMERO: ¿la fase 1 supera el azar EN SU PROPIO DOMINIO? azar = 1/3 =  33.33%
  (fase 1 entrena sobre 48 bolsas de entrenamiento; el acierto se lee sobre 18 bolsas de evaluación de la fuente)

  familia   norm       brazo    acierto fuente   ¿supera el azar?
  lunas     ninguna    TGL     77.78% ± 0.000                 sí
  lunas     batchnorm  TGL     66.67% ± 0.000                 sí
  lunas     layernorm  TGL     77.78% ± 0.000                 sí
  toroide   ninguna    TGL    100.00% ± 0.000                 sí
  toroide   batchnorm  TGL     94.44% ± 0.000                 sí
  toroide   layernorm  TGL    100.00% ± 0.000                 sí
  rollo     ninguna    TGL    100.00% ± 0.000                 sí
  rollo     batchnorm  TGL    100.00% ± 0.000                 sí
  rollo     layernorm  TGL    100.00% ± 0.000                 sí

9 de 9 celdas superan el azar en la fuente.

el acierto del BLANCO al lado, sobre las 1 semillas de reporte --- y nada contra qué
leerlo: el piso se retiró

## El artefacto, y la vuelta por disco

`guardar` escribe `banderas.json` —cada bandera de la corrida, para que una gráfica
pueda decir bajo qué se midió—, `barrido.json` —las celdas, las filas, los
hiperparámetros que cada búsqueda eligió y los ensayos de las dos— y un
`<familia>.npz` por familia. **Los bloques `escalones` y `pisos` ya no están**, y no
están vacíos: una lista vacía se lee como «se midió y no había nada», y lo que es
cierto es que no se puede medir nada. Es la forma de `CREDA.toy_experiment.guardar`, con el
contenido de MIL-CREDA.

Y después **se lee de vuelta**, desde los archivos reales. `np.savez` no falla con un
valor que no puede serializar sin pickle: lo guarda y revienta al cargarlo, en el otro
cuaderno y sin ninguna pista de cuál era la clave. Probar las dos mitades contra un
diccionario escrito a mano verifica las dos mitades y nunca la unión, que es lo único de
lo que se trataba.

**Y desde `LATENT_MEMBERS`, el `.npz` lleva además las POSICIONES.** Por celda:
la representación euclidiana de cada bolsa —la de la Ec. 19, la combinación convexa
sobre la que opera el clasificador—, de los dos dominios, con su clase y su índice, y
el blanco en los dos momentos: antes de la fase 2, cuando `theta_t` todavía es
`theta_s` bolsa por bolsa, y después. Son tres vistas y no una porque la pregunta que
el cuaderno de gráficas le hace a esta figura es qué le hizo la fase 2 al codificador
del blanco, y una sola nube no la contesta. La fuente va una vez porque no se mueve, y
que no se mueva es una medición: `latentSourceMoved` viaja en cada celda al lado de
`latentTargetMoved`.

**Lo que NO se escribe, y por qué, para que la omisión sea una decisión.** Las
posiciones de las INSTANCIAS no viajan: la figura dibuja bolsas, así que nadie las
leería; son treinta veces los bytes por vista; y los pesos `beta` que ubican a una
bolsa dentro de su propia nube de instancias tampoco viajan, de modo que la nube
llegaría sin lo único que la haría legible. Tampoco viaja una predicción ni una
confianza por punto: la figura ya gasta el color en la clase y la marca en el dominio,
y `toy_bags` tiene medido lo que pasa cuando a un panel de esta cardinalidad se le
pide un tercer canal —a 12 bolsas por clase el canal del rótulo dejó de leerse—.
Agregar cualquiera de las dos cosas es una línea acá y una vista más en esa tupla.

Y el cuaderno de al lado, `MIL-CREDA/Notebooks/Toyset_Plot.ipynb`, **lee este
artefacto**: abre lo que `guardar` dejó, arma sus tablas con `toy_tables` y dibuja el
espacio latente de cada familia con `toy_run.animar_familia`. Entrenar y dibujar están
separados a propósito, y el costo de no separarlos está escrito en el propio CREDA:
antes de ese corte, retocar un color costaba el barrido entero.

In [6]:
ESCRITOS = toy_run.guardar(DESTINO, BARRIDO)
print("escrito en", DESTINO.relative_to(REPOSITORY))
for _r in ESCRITOS:
    print(f"  {_r.name:24s} {_r.stat().st_size:>9,} bytes")
print()

LEIDO = toy_run.cargar(DESTINO)
print("leído de vuelta desde los archivos reales:")
print("  banderas: escala", LEIDO["flags"]["scale"]["name"],
      "/ revisión", LEIDO["flags"]["revision"],
      "/ ensayo", LEIDO["flags"]["scale"]["isPilot"])
print("  celdas   ", len(LEIDO["cells"]), "de", len(BARRIDO["cells"]))
print("  filas    ", len(LEIDO["rows"]), "de", len(BARRIDO["rows"]))
assert "rungs" not in LEIDO and "floors" not in LEIDO, (
    "la vuelta reinstaló un escalón o un piso")
print("  escalones/pisos: ausentes, no vacíos (un brazo: no hay par que restar)")
print("  familias ", sorted(LEIDO["families"]))
for _f, _p in sorted(LEIDO["families"].items()):
    print(f"    {_f}: {len(_p['info'])} registros, "
          f"{len(_p['hist_s'])} historias de fase 1, "
          f"{len(_p['sigma'])} grillas de sigma, "
          f"{len(_p['latent'])} juegos de posiciones")
print()
# Lo que las posiciones cuestan, medido sobre lo que acaba de escribirse y no
# citado de otra corrida: el tamaño del artefacto es lo que decide si una vista
# más es barata o no, y esa decisión se toma leyendo este número.
_bytes_latentes = sum(
    _a.nbytes for _p in LEIDO["families"].values()
    for _miembros in _p["latent"].values() for _a in _miembros.values())
_bytes_npz = sum(_r.stat().st_size for _r in ESCRITOS if _r.suffix == ".npz")
print("lo que ocupan las posiciones:")
print(f"  miembros declarados:   {', '.join(toy_run.LATENT_MEMBERS)}")
print(f"  en memoria, sin comprimir: {_bytes_latentes:,} bytes")
print(f"  los `.npz` en disco:       {_bytes_npz:,} bytes "
      f"({len(LEIDO['families'])} familias, comprimidos)")
assert all(set(_m) == set(toy_run.LATENT_MEMBERS)
           for _p in LEIDO["families"].values() for _m in _p["latent"].values()), (
    "una celda volvió de disco sin alguno de los miembros declarados")
print()
assert set(LEIDO["cells"]) == set(BARRIDO["cells"]), "la vuelta perdió celdas"
assert LEIDO["hp1"] == BARRIDO["hp1"], "la vuelta perdió la elección de fase 1"
assert LEIDO["scale"] == ESCALA.stamp(), "la vuelta perdió el sello de la escala"
print("la vuelta cierra: las claves que volvieron son las que viajaron.")
print()
print("el sello de la escala que lleva CADA registro:")
print(" ", json.dumps(LEIDO["scale"]))

escrito en MIL-CREDA/Results/Toyset/pilot
  banderas.json               28,845 bytes
  barrido.json               157,355 bytes
  lunas.npz                  490,453 bytes
  toroide.npz                490,342 bytes
  rollo.npz                  490,087 bytes

leído de vuelta desde los archivos reales:
  banderas: escala pilot / revisión research-concept-r34.md / ensayo True
  celdas    9 de 9
  filas     9 de 9
  escalones/pisos: ausentes, no vacíos (un brazo: no hay par que restar)
  familias  ['lunas', 'rollo', 'toroide']
    lunas: 3 registros, 3 historias de fase 1, 3 grillas de sigma, 3 juegos de posiciones
    rollo: 3 registros, 3 historias de fase 1, 3 grillas de sigma, 3 juegos de posiciones
    toroide: 3 registros, 3 historias de fase 1, 3 grillas de sigma, 3 juegos de posiciones

lo que ocupan las posiciones:
  miembros declarados:   source, targetBefore, targetAfter, classSource, classTarget, bagSource, bagTarget, trajectorySteps, trajectoryTarget, trajectoryPrediction, pred

## La trayectoria, y lo que cuesta guardarla

La figura por dataset del cuaderno de CREDA es **animada**: sus cuadros son los
pasos de la fase 2, no dos momentos. Guardar una posición por bolsa en cada paso
multiplica el artefacto, así que la **zancada está declarada** y no es un
accidente: se guarda un cuadro cada `k` pasos, más el paso 0 y el último
*siempre*. El cero es el punto de control donde `theta_t` es todavía `theta_s`
—el mismo que el primer escalón de la tabla de intervención lee— y el último es
donde la parada temprana dejó la corrida, que casi nunca cae en un múltiplo de
`k`.

`k` sale de un número de cuadros declarado (`TRAJECTORY_FRAMES`) y de los pasos
que la escala declara, y no de un literal: con el 10 de CREDA escrito a mano, la
escala de ensayo se quedaría con los cuadros que su propio techo dejara caer en
esa zancada y estaría ensayando una figura que la corrida completa no dibuja. La
celda de abajo imprime `k`, los cuadros que salieron y el costo, para esta
corrida y proyectado a la escala completa.

Lo que sigue es el costo **de esta corrida**, medido sobre lo que acaba de
quedar en disco, no la cifra de otra sesión.

In [7]:
# El costo de la trayectoria, medido sobre los archivos que `guardar` acaba de
# escribir y proyectado a la escala completa con la misma aritmética. Una cifra
# citada de otra sesión es exactamente la clase de número que deja de ser cierto
# sin que nada falle.
# La dimensión la ELIGE la búsqueda (`dim`, fase 1), así que se lee de las
# celdas que acaban de correr y no de `config.MLP_OUTPUT_DIM`. Esa constante
# vale 16 y esta corrida eligió otra cosa: imprimirla acá daba una cifra
# equivocada por el factor entre las dos, que es exactamente una constante
# declarada haciéndose pasar por un valor buscado.
_dims = sorted({_f["hyper"]["dim"] for _f in BARRIDO["cells"].values()})
_bytes_de = {_d: ESCALA.bags_total * _d * 4 + ESCALA.bags_total * 2
             for _d in _dims}
_bytes_por_cuadro = max(_bytes_de.values())
_bytes_declarado = (ESCALA.bags_total * config.MLP_OUTPUT_DIM * 4
                    + ESCALA.bags_total * 2)
_cuadros = sorted({fila["trajectoryFrames"] for fila in BARRIDO["cells"].values()})
# La rejilla se DECLARA en epocas y los pasos se derivan, asi que se
# imprimen las dos: un numero rotulado con la unidad de la otra es como se
# lee una figura que miente sobre su propio eje.
_zancadas = sorted({fila["frameStride"] for fila in BARRIDO["cells"].values()})
_zancadas_ep = sorted({fila["frameEpochs"] for fila in BARRIDO["cells"].values()})
_manijas = len(BARRIDO["cells"])
_npz = sum(_r.stat().st_size for _r in ESCRITOS if _r.suffix == ".npz")

_zancada_completa = toy_run.frame_stride(toy_run.FULL)
_zancada_completa_ep = toy_run.frame_epochs(toy_run.FULL)
_pasos_completa = toy_run.FULL.epochs_2 * toy_run.STEPS_PER_EPOCH_DECLARED
_cuadros_completa = len(toy_run.frame_steps(toy_run.FULL, _pasos_completa))
_manijas_completa = (len(toy_run.FULL.report_seeds) * len(toy_run.NORMALIZATIONS)
                     * len(BARRIDO["flags"]["families"]))

print("ESTA corrida")
print(f"  cuadros por celda:        {_cuadros}")
print(f"  zancada declarada:        {_zancadas_ep} epocas = {_zancadas} pasos "
      f"(de {ESCALA.epochs_2} epocas "
      f"y {toy_run.TRAJECTORY_FRAMES} cuadros pedidos)")
print(f"  dimensión elegida:        {_dims} (la busca la fase 1; "
      f"`config.MLP_OUTPUT_DIM` declara {config.MLP_OUTPUT_DIM})")
print(f"  bytes por cuadro:         {_bytes_por_cuadro:,} "
      f"({ESCALA.bags_total} bolsas x {max(_dims)} dimensiones x 4 "
      f"+ una decisión por bolsa)")
if _bytes_declarado != _bytes_por_cuadro:
    print(f"    con la constante declarada habrían salido "
          f"{_bytes_declarado:,} bytes, "
          f"{_bytes_por_cuadro / _bytes_declarado:.1f}x menos que lo real")
if len(_dims) > 1:
    print(f"    (las celdas no eligieron todas la misma: {_bytes_de}; "
          f"la proyección de abajo usa la mayor)")
print(f"  celdas escritas:          {_manijas}")
print(f"  los .npz en disco suman:  {_npz:,} bytes")
print()
print("la escala COMPLETA, proyectada con la misma aritmética (no se corre acá)")
print(f"  zancada:                  {_zancada_completa_ep} epocas = "
      f"{_zancada_completa} pasos "
      f"(de {_pasos_completa} pasos)")
print(f"  cuadros por celda:        {_cuadros_completa}")
print(f"  celdas:                   {_manijas_completa}")
print(f"  trayectoria sin comprimir: "
      f"{_cuadros_completa * _bytes_por_cuadro * _manijas_completa:,} bytes")
print(f"  y SIN zancada declarada:   "
      f"{(_pasos_completa + 1) * _bytes_por_cuadro * _manijas_completa:,} "
      f"bytes, con {_pasos_completa + 1} cuadros por celda que dibujar")

ESTA corrida
  cuadros por celda:        [5]
  zancada declarada:        [1] epocas = [16] pasos (de 4 epocas y 20 cuadros pedidos)
  dimensión elegida:        [64] (la busca la fase 1; `config.MLP_OUTPUT_DIM` declara 16)
  bytes por cuadro:         20,898 (81 bolsas x 64 dimensiones x 4 + una decisión por bolsa)
    con la constante declarada habrían salido 5,346 bytes, 3.9x menos que lo real
  celdas escritas:          9
  los .npz en disco suman:  1,470,882 bytes

la escala COMPLETA, proyectada con la misma aritmética (no se corre acá)
  zancada:                  5 epocas = 80 pasos (de 1600 pasos)
  cuadros por celda:        21
  celdas:                   90
  trayectoria sin comprimir: 39,497,220 bytes
  y SIN zancada declarada:   3,011,192,820 bytes, con 1601 cuadros por celda que dibujar


## El sello

Contra qué código corrió este cuaderno. Sin él, una corrida vieja y una recién hecha se
ven idénticas y la vieja se sigue creyendo. Cubre todo `src/`, así que un cambio en
cualquier módulo lo mueve —es más grueso que lo que este cuaderno lee, y eso es
deliberado: una regla más fina es una regla que dos implementaciones del sello pueden
entender distinto.

In [8]:
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())

SOURCES-SHA256 5a98f4b66db3474f02bdface4c75c249fe45535952343beb09f38e37617ea4f0
